In [1]:
import json
import re
import time
import random
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager

# =================設定區 (Configuration)=================

# 模擬真人行為的設定
def random_sleep(min_seconds=2, max_seconds=5):
    """隨機等待一段時間"""
    time.sleep(random.uniform(min_seconds, max_seconds))

def simulate_mouse_movement(driver):
    """模擬滑鼠隨機移動"""
    try:
        action = ActionChains(driver)
        # 隨機移動幾次
        for _ in range(random.randint(1, 3)):
            x_offset = random.randint(-50, 50)
            y_offset = random.randint(-50, 50)
            action.move_by_offset(x_offset, y_offset).perform()
            time.sleep(random.uniform(0.1, 0.5))
    except Exception:
        pass # 忽略移動錯誤

def normalize_time(time_str):
    """
    將各家新聞網的時間格式統一轉為 YYYY-MM-DD HH:MM
    這裡需要根據實際抓到的字串做調整
    """
    if not time_str:
        return ""
    
    time_str = time_str.strip()
    try:
        # 輸入: "2025-11-28T11:55:00+08:00" 輸出: "2025/11/28 11:55"
        if 'T' in time_str:
            # 依據 'T' 切割，取前半部日期，再取後半部時間的前5個字(HH:MM)
            date_part = time_str.split('T')[0]
            time_part = time_str.split('T')[1][:5]
            time_str = f"{date_part} {time_part}"
        if '年' in time_str:
            time_str = time_str.replace('年', '/').replace('月', '/').replace('日', '')
        if '-' in time_str:
            time_str = time_str.replace('-', '/')
        # ===============================================
        # 常見格式處理 (依需求擴充)
        # 範例：2023/11/26 14:00, 2023-11-26 14:00
        # 簡單過濾掉中文如 "發布時間："
        for char in ['發布時間：', '出版：', '時間：', '報導','記者','發佈','最後更新',]:
            time_str = time_str.replace(char, '')
        
        return time_str.strip()
    except Exception as e:
        return time_str # 無法轉換則回傳原字串

# =================網站選擇器設定 (請依實際網頁檢查結果修改)=================
# 這是您需要透過 F12 (檢查) 確認的部分。我已填入常見的選擇器作為預設值。
SITES_CONFIG = [
    {
        "name": "TVBS",
        "url": "https://news.tvbs.com.tw/hot",
        "list_selector": ".news_list .list li > a:nth-of-type(1) ",  # 列表頁的新聞連結
        "title_selector": "h1.title",                 # 內文標題
        "category_selector": ".bread a:nth-of-type(2)", # 內文分類
        "time_selector": ".author",                   # 內文時間
        "content_selector": "#news_detail_div"        # 內文內容
    },
    {
        "name": "UDN",
        "url": "https://udn.com/rank/pv/2",
        "list_selector": ".story-list__news h2 a",
        "title_selector": "h1.article-content__title",
        "category_selector": ".article-content__breadcrumb a:nth-of-type(2)",
        "time_selector": ".article-content__time",
        "content_selector": ".article-content__editor"
    },
    {
        "name": "ChinaTimes",
        "url": "https://www.chinatimes.com/hotnews/?chdtv",
        "list_selector": ".vertical-list li h3 a",
        "title_selector": ".article-header h1",
        "category_selector": ".breadcrumb li:last-child span[itemprop='name']",
        "time_selector": ".meta-info time",
        "content_selector": ".article-body"
    },
    {
        "name": "LTN",
        "url": "https://news.ltn.com.tw/list/breakingnews/popular",
        "list_selector": "ul.list li > a",
        # 標題：涵蓋主網、體育、娛樂、汽車、3C
        "title_selector": ".whitecon h1, .article h1, .news_content h1, .topic_title h1, .news_title h1, h1",
        # 分類
        "category_selector": ".breadcrumbs a:last-child, .boxTitle .guide a:last-child, .news_content .guide a:last-child, .guide a:last-child",
        # 時間：涵蓋所有已知的子網域 CSS
        "time_selector": ".whitecon .time, .article .time, span.article_time, .news_content .date, .c_time, .writer_date, .mobile_none",
        # 內文：涵蓋主網、體育(.news_p)、娛樂(.news_content)、財經(.text)
        "content_selector": ".whitecon .text, .article .text, .text, .news_p, .news_content, div[itemprop='articleBody']"
    },
    {
        "name": "SETN",
        "url": "https://www.setn.com/viewall.aspx?pagegroupid=0",
        "list_selector": ".NewsList .newsItems h3 a",
        # 主網站,子網站
        "title_selector": "h1.news-title-3, h1",
        "category_selector": ".newsBreadcrumb .breadcrumb a:last-child",
        "time_selector": ".page_date, .newsTime time",
        "content_selector": "article"
    },
    {
        "name": "ETtoday",
        "url": "https://www.ettoday.net/news/hot-news.htm",
        "list_selector": ".part_pictxt_3 .piece h3 a",
        "title_selector": "h1.title",
        "category_selector": ".menu_bread_crumb div[itemprop='itemListElement']:last-of-type span[itemprop='name']",
        "time_selector": ".date",
        "content_selector": ".story"
    }
]

# =================輔助函式：建立 Driver (含防呆設定)=================
def get_driver():
    """建立並回傳一個新的 Driver 實例，設定好防卡死參數"""
    chrome_options = Options()
    # chrome_options.add_argument("--headless") # 除錯時建議先註解掉，確認沒問題再開
    chrome_options.add_argument("--headless=new") # 建議使用新版無頭模式
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    # 關鍵：設定頁面載入策略為 'eager' (不等圖片廣告，載入 HTML 就回傳)
    chrome_options.page_load_strategy = 'eager' 

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    # 關鍵：強制設定頁面載入超時為 20 秒 (避免卡死 120 秒)
    driver.set_page_load_timeout(20)
    
    return driver

# =================主程式=================

def start_crawler():
    # 1. 取得初始 Driver
    print("正在啟動瀏覽器...")
    driver = get_driver()
    
    all_news_data = []

    try:
        for site in SITES_CONFIG:
            print(f"正在爬取列表：{site['name']} - {site['url']}")
            
            # --- 列表頁抓取階段 ---
            try:
                driver.get(site['url'])
                random_sleep(3, 5) # 等待列表頁載入
                
                # 使用 BeautifulSoup 解析列表頁
                soup_list = BeautifulSoup(driver.page_source, 'html.parser')
                links = soup_list.select(site['list_selector'])
                
                news_urls = []
                # 取前 5 篇作為測試 (若要全部請拿掉 [:5])
                for link in links[:50]:
                    href = link.get('href')
                    if href:
                        # 處理相對路徑
                        if not href.startswith('http'):
                            if site['name'] == 'TVBS': href = 'https://news.tvbs.com.tw' + href
                            elif site['name'] == 'LTN': href = href 
                            elif site['name'] == 'SETN': href = 'https://www.setn.com/' + href
                            elif site['name'] == 'UDN': href = 'https://udn.com' + href
                            # 其他相對路徑補齊邏輯...
                        news_urls.append(href)
                
                print(f"取得 {len(news_urls)} 篇連結，開始逐篇抓取內容...")

            except Exception as e:
                print(f"❌ 列表頁抓取失敗 ({site['name']}): {e}")
                # 列表頁都掛了，嘗試重啟 driver 保險起見
                try: driver.quit()
                except: pass
                driver = get_driver()
                continue

            # --- 內文頁抓取階段 ---
            for url in news_urls:
                try:
                    driver.get(url)
                    simulate_mouse_movement(driver) # 模擬滑鼠
                    random_sleep(2, 4) # 模擬閱讀時間

                    # 解析內文
                    soup_article = BeautifulSoup(driver.page_source, 'html.parser')
                    
                    # 1. 抓取標題
                    title = "N/A"
                    title_tag = soup_article.select_one(site['title_selector'])
                    if title_tag: title = title_tag.get_text(strip=True)

                    # 2. 抓取分類
                    category = "N/A"
                    cat_tag = soup_article.select_one(site['category_selector'])
                    if cat_tag: category = cat_tag.get_text(strip=True)

                    # 3. 抓取時間 (整合 CSS / HTML5 datetime / Meta Tag)
                    pub_time = "N/A"
                    time_tag = soup_article.select_one(site['time_selector'])
                    
                    # A. 優先嘗試 CSS 與屬性
                    if time_tag:
                        if time_tag.get('datetime'):
                            pub_time = normalize_time(time_tag.get('datetime'))
                        elif site['name'] == 'TVBS':
                            for text_part in time_tag.stripped_strings:
                                if "發佈時間" in text_part:
                                    pub_time = normalize_time(text_part)
                                    break
                        else:
                            pub_time = normalize_time(time_tag.get_text(strip=True))
                    
                    # B. 救援模式：如果上面沒抓到，嘗試抓 Meta Tag (LTN/中時 有效)
                    if (pub_time == "N/A" or pub_time == ""):
                        meta_time = soup_article.select_one('meta[property="article:published_time"]') or \
                                    soup_article.select_one('meta[name="pubdate"]')
                        if meta_time:
                            pub_time = normalize_time(meta_time.get('content'))

                    # 4. 抓取內文 (整合 CSS / JSON-LD)
                    content = "N/A"
                    content_tag = soup_article.select_one(site['content_selector'])
                    
                    # A. 優先嘗試 CSS
                    if content_tag: 
                        content = content_tag.get_text(strip=True)
                    
                    # B. 救援模式：嘗試 JSON-LD (針對 LTN 子網域 / 中時財經)
                    if (content == "N/A" or content == ""):
                        json_scripts = soup_article.select('script[type="application/ld+json"]')
                        for script in json_scripts:
                            try:
                                data = json.loads(script.get_text())
                                if isinstance(data, list): data = data[0]
                                if 'articleBody' in data:
                                    content = data['articleBody']
                                    # 順便補救時間
                                    if pub_time == "N/A" and 'datePublished' in data:
                                        pub_time = normalize_time(data['datePublished'])
                                    break
                            except:
                                continue

                    # 5. 抓取時間 (當下)
                    fetch_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    print(f"成功抓取: {title[:10]}...")

                    all_news_data.append({
                        "Site": site['name'],
                        "URL": url,
                        "Category": category,
                        "Publish_Time": pub_time,
                        "Fetch_Time": fetch_time,
                        "Title": title,
                        "Content": content
                    })

                except Exception as e:
                    print(f"⚠️ 抓取單篇新聞失敗 ({url}): {e}")
                    
                    # === 自動重啟機制 ===
                    # 檢測是否為連線超時或瀏覽器卡死
                    error_msg = str(e).lower()
                    if any(x in error_msg for x in ['timeout', 'connection', 'refused', 'reset', 'disconnected', 'window']):
                        print("🔴 偵測到瀏覽器異常，正在重啟 Driver...")
                        try: driver.quit()
                        except: pass
                        time.sleep(3)
                        driver = get_driver() # 重新建立全新的瀏覽器
                        print("🟢 瀏覽器重啟完成")
                    # ==================
                    continue
            
            print(f"{site['name']} 完成。")
            random_sleep(4, 5) # 網站間的休息時間

    except Exception as e:
        print(f"爬蟲發生致命錯誤: {e}")
    finally:
        try: driver.quit()
        except: pass

    # 匯出 CSV
    df = pd.DataFrame(all_news_data)
    filename = f"news_hot_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df.to_csv(filename, index=False, encoding='utf-8-sig') 
    print(f"資料已匯出至 {filename}")

if __name__ == "__main__":
    start_crawler()

正在啟動瀏覽器...
正在爬取列表：TVBS - https://news.tvbs.com.tw/hot
取得 50 篇連結，開始逐篇抓取內容...
成功抓取: 桃園「停課92班」！...
成功抓取: 寶雅「發票中200萬...
成功抓取: 朱孝天遭除名F4開嗆...
成功抓取: 遭爆國小起霸凌同學還...
成功抓取: 天氣／4縣市大雨特報...
成功抓取: 如黑幫電影！高雄27...
成功抓取: 北市議員陳怡君涉貪4...
成功抓取: 小煜5年婚告吹「兩大...
成功抓取: 公車司機「暈眩2分鐘...
成功抓取: 立威廉罹甲狀腺癌　名...
成功抓取: 賴瑞隆恐退選？爆民進...
成功抓取: 食品巨頭重大改革！百...
成功抓取: 內湖深夜火警！公寓4...
成功抓取: 日本青森縣外海7.6...
成功抓取: 再爆涉貪關鍵「恐重判...
成功抓取: 東京若爆規模7地震　...
成功抓取: 日本青森外海7.5強...
成功抓取: 台灣最新詐騙！收到1...
成功抓取: 有片／垃圾車變移動炸...
成功抓取: 青森地震！日示警恐爆...
成功抓取: 川普點頭准了！同意輝...
成功抓取: 看《火影忍者》想脫北...
成功抓取: 峮峮退出《飢餓》發聲...
成功抓取: 金球獎入圍／李奧納多...
成功抓取: 火鍋花生粉驗出黃麴毒...
成功抓取: 機率升至1%！日本氣...
成功抓取: 朱孝天入行25年不懂...
成功抓取: ELLA大推「超慢跑...
成功抓取: 日本青森外海7.5強...
成功抓取: 離婚正妹護理師！小煜...
成功抓取: Fed會議倒數！黃金...
成功抓取: 青森7.5強震30傷...
成功抓取: 慢性腎臟病3年惡化！...
成功抓取: 胡瓜出事了！主持一半...
成功抓取: 賈永婕曝夫妻放風狂歡...
成功抓取: 金錢觀差超大！4星座...
成功抓取: 青森深夜7.5強震！...
成功抓取: 王柏傑、謝欣穎分手半...
成功抓取: 賴瑞隆家人爆霸凌下一...
成功抓取: 通緝犯羅天義遭電眼監...
成功抓取: 確定了？美媒爆「佐佐...
成功抓取: 有片／高雄水泥車右轉...
成功抓取: 秦嵐、魏大勛姊弟戀告...
成功抓取: 《所得稅法》三讀　長...
成功抓取: 分析／民眾黨力拱「兩...
成功抓取: B流小心誤認腸胃炎　...
成功抓取